<a href="https://colab.research.google.com/github/Seanm18/proyectos_portafolio/blob/main/chatbot_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Creamos un documento de prueba sobre Utah
contenido = """
Utah es un estado en el oeste de Estados Unidos.
Su capital es Salt Lake City.
La población es de aproximadamente 3.3 millones de personas.
El sector tecnológico de Utah se conoce como Silicon Slopes.
Las principales empresas tech en Utah son Adobe, Qualtrics y Domo.
El salario promedio en IT en Utah es de $85,000 al año.
Utah tiene un clima desértico con inviernos nevados en las montañas.
Park City es famosa por sus pistas de ski de clase mundial.
"""

with open('utah_info.txt', 'w') as f:
    f.write(contenido)

print("Archivo creado exitosamente")

Archivo creado exitosamente


In [7]:
!pip install groq

import os
from groq import Groq
from google.colab import userdata

groq_api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=groq_api_key)

# Leer el documento
with open('utah_info.txt', 'r') as f:
    texto = f.read()

print("Documento cargado exitosamente")
print(texto)

Documento cargado exitosamente

Utah es un estado en el oeste de Estados Unidos.
Su capital es Salt Lake City.
La población es de aproximadamente 3.3 millones de personas.
El sector tecnológico de Utah se conoce como Silicon Slopes.
Las principales empresas tech en Utah son Adobe, Qualtrics y Domo.
El salario promedio en IT en Utah es de $85,000 al año.
Utah tiene un clima desértico con inviernos nevados en las montañas.
Park City es famosa por sus pistas de ski de clase mundial.



In [11]:
# Dividir el documento en fragmentos pequeños
fragmentos = [linea.strip() for linea in texto.split('\n') if linea.strip()]

print(f"Total de fragmentos: {len(fragmentos)}")
for i, f in enumerate(fragmentos):
    print(f"{i}: {f}")

Total de fragmentos: 8
0: Utah es un estado en el oeste de Estados Unidos.
1: Su capital es Salt Lake City.
2: La población es de aproximadamente 3.3 millones de personas.
3: El sector tecnológico de Utah se conoce como Silicon Slopes.
4: Las principales empresas tech en Utah son Adobe, Qualtrics y Domo.
5: El salario promedio en IT en Utah es de $85,000 al año.
6: Utah tiene un clima desértico con inviernos nevados en las montañas.
7: Park City es famosa por sus pistas de ski de clase mundial.


In [12]:
!pip install langchain langchain-google-genai chromadb pypdf

import chromadb

cliente = chromadb.Client()

try:
    cliente.delete_collection("utah_docs")
except Exception:
    pass

coleccion = cliente.create_collection("utah_docs")

# Guardar fragmentos en ChromaDB
coleccion.add(
    documents=fragmentos,
    ids=[f"doc_{i}" for i in range(len(fragmentos))]
)

print(f"✅ {len(fragmentos)} fragmentos guardados en ChromaDB")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 68.5MiB/s]


✅ 8 fragmentos guardados en ChromaDB


In [ ]:
from groq import Groq
from google.colab import userdata

groq_api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=groq_api_key)

def preguntar(pregunta):
    # 1. Buscar fragmentos relevantes en ChromaDB
    resultados = coleccion.query(
        query_texts=[pregunta],
        n_results=3
    )

    contexto = "\n".join(resultados['documents'][0])

    # 2. Enviar contexto + pregunta a Groq
    respuesta = cliente_groq.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "user",
                "content": f"Basándote SOLO en esta información:\n{contexto}\n\nResponde: {pregunta}"
            }
        ]
    )

    return respuesta.choices[0].message.content

# Prueba
print(preguntar("¿Cuál es la capital de Utah?"))

La capital de Utah es Salt Lake City.


In [ ]:
from groq import Groq
from google.colab import userdata

groq_api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=groq_api_key)

# Test simple sin ChromaDB
respuesta = cliente_groq.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Di hola"}]
)

print(respuesta.choices[0].message.content)

Hola, ¿en qué puedo ayudarte?


In [ ]:
def preguntar(pregunta):
    # 1. Buscar fragmentos relevantes en ChromaDB
    resultados = coleccion.query(
        query_texts=[pregunta],
        n_results=3
    )

    contexto = "\n".join(resultados['documents'][0])

    # 2. Enviar contexto + pregunta a Groq
    respuesta = cliente_groq.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "user",
                "content": f"Basándote SOLO en esta información:\n{contexto}\n\nResponde: {pregunta}"
            }
        ]
    )

    return respuesta.choices[0].message.content

# Prueba con el documento de Utah
print(preguntar("¿Cuál es la capital de Utah?"))
print(preguntar("¿Cuánto gana un IT en Utah?"))
print(preguntar("¿Qué empresas tech hay en Utah?"))


#Prueba de información externa
print(preguntar("¿Cuál es el mejor restaurante en Salt Lake City?"))

La capital de Utah es Salt Lake City.
Según la información proporcionada, un IT en Utah gana $85,000 al año.
Según la información proporcionada, las principales empresas tech en Utah son:

1. Adobe
2. Qualtrics
3. Domo

Estas empresas forman parte del sector tecnológico conocido como Silicon Slopes.
No puedo responder a tu pregunta con precisión por que la información proporcionada solo contiene información sobre el estado de Utah y su capital, pero no sobre restaurantes.


In [ ]:
# Chatbot interactivo
print("Chatbot de Utah listo. Escribe 'salir' para terminar.\n")

while True:
    pregunta = input("Tú: ")
    if pregunta.lower() == 'salir':
        print("¡Hasta luego!")
        break
    respuesta = preguntar(pregunta)
    print(f"Bot: {respuesta}\n")


🤖 Chatbot de Utah listo. Escribe 'salir' para terminar.

Tú: hola
Bot: Hola! Me alegra charlar sobre Utah. ¿Quieres saber más sobre este hermoso estado?

Tú: si, ¿cual es la capital?
Bot: La capital es Salt Lake City.

Tú: dime mas acerca de esto
Bot: Con esta información limitada, puedo proporcionarte algunas observaciones y preguntas:

**Observaciones:**

- El salario promedio en IT en Utah es alto para una región en particular, lo que sugiere una alta demanda de profesionales en tecnología.
- La población de Utah es significativa, lo que abre oportunidades laborales en sectores como IT.

**Preguntas:**

- ¿Cuáles son las industrias más prominentes en Utah que requieren perfiles en tecnología?
- ¿Cuál es la distribución de población en Utah? ¿Hay áreas urbanas o rurales que podrían estar más relacionadas con la industria IT?
- ¿Existe un clima laboral favorable en Utah que atraiga a profesionales en tecnología?

**Posibles industrias relacionadas con IT en Utah:**

- Según la Oficina